In [1]:
# Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# For Visualization
import folium
from folium.plugins import MarkerCluster

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set(style="whitegrid")

In [21]:
# Loading the Cleaned Dataset
data_path = '../data/Global_Terrorism_Database_May_2022.csv'
df = pd.read_csv(data_path)

# Display the first few rows
df.head()

,eventid,iyear,imonth,iday,approxdate,extended,resolution,country,country_txt,region,...,addnotes,scite1,scite2,scite3,dbsource,INT_LOG,INT_IDEO,INT_MISC,INT_ANY,related
0,197000000001,1970,7,2,NaN,0,NaN,58,Dominican Republic,2,...,NaN,NaN,NaN,NaN,PGIS,0,0,0,0,NaN
1,197000000002,1970,0,0,NaN,0,NaN,130,Mexico,1,...,NaN,NaN,NaN,NaN,PGIS,0,1,1,1,NaN
2,197001000001,1970,1,0,NaN,0,NaN,160,Philippines,5,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
3,197001000002,1970,1,0,NaN,0,NaN,78,Greece,8,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
4,197001000003,1970,1,0,NaN,0,NaN,101,Japan,4,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN


In [22]:
# Remove Records with Missing Latitude or Longitude
df_clean = df.dropna(subset=['latitude', 'longitude'])

# Verify Removal
print(f"Total records after removal: {df_clean.shape[0]}")

Total records after removal: 205014


In [29]:
# Selecting Relevant Features
features = ['gname', 'latitude', 'longitude']
df_groups = df_clean[features]

In [30]:
# Aggregating the Number of Attacks per Group and Location
attack_counts_group = df_groups.groupby(['gname', 'latitude', 'longitude']).size().reset_index(name='count')

# Displaying Sample Aggregated Data
attack_counts_group.head()

,gname,latitude,longitude,count
0,1 May,37.997490,23.762728,9
1,14 K Triad,22.192266,113.547631,4
2,14 March Coalition,34.438094,35.830837,1
3,14th of December Command,-33.366238,-70.505302,3
4,15th of September Liberation Legion,9.933333,-84.083333,1


In [32]:
# Assign Unique Colors to Each Group
import numpy as np

# Get unique groups
unique_groups = attack_counts_group['gname'].unique()

# Generate a color palette
colors = sns.color_palette('hls', len(unique_groups)).as_hex()

# Create a dictionary mapping groups to colors
group_colors = dict(zip(unique_groups, colors))

# Display the color mapping
group_colors

{'1 May': '#db5f57',
 '14 K Triad': '#db5f57',
 '14 March Coalition': '#db5f57',
 '14th of December Command': '#db5f57',
 '15th of September Liberation Legion': '#db6057',
 '16 January Organization for the Liberation of Tripoli': '#db6057',
 '1920 Revolution Brigades': '#db6057',
 '1st of May Group': '#db6057',
 '2 April Group': '#db6057',
 '20 December Movement (M-20)': '#db6157',
 '22 May 1948': '#db6157',
 '23 May Democratic Alliance (Algeria)': '#db6157',
 '23rd of September Communist League': '#db6157',
 '28 February Armed Group': '#db6157',
 '28 May Armenian Organization': '#db6257',
 '28s': '#db6257',
 '28th of December Group': '#db6257',
 '2nd of June Movement': '#db6257',
 "31 January People's Front (FP-31)": '#db6357',
 '313 Brigade (Syria)': '#db6357',
 '4 August National Organization': '#db6357',
 '7 April Libyan Organization': '#db6357',
 '9 February': '#db6357',
 "9 May People's Liberation Force": '#db6457',
 'A Resistance Group': '#db6457',
 "A'chik Matgrik Elite Force (

In [44]:
import folium
from folium.plugins import MarkerCluster

# Create a base map
m = folium.Map(location=[0, 0], zoom_start=2)

# Create a marker cluster
marker_cluster = MarkerCluster().add_to(m)

# Add markers to the cluster
for idx, row in attack_counts_group.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5 + row['count'] / 100,  # Adjust the radius based on the count
        color=group_colors[row['gname']],
        fill=True,
        fill_color=group_colors[row['gname']],
        fill_opacity=0.6,
        popup=f"Group: {row['gname']}<br>Count: {row['count']}"
    ).add_to(marker_cluster)

# Save the map to an HTML file
m.save('terrorism_map.html')

